In [9]:
from dotenv import load_dotenv
import os

from langchain_groq import ChatGroq

load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.getenv("GROQ_API_KEY")
)

response = llm.invoke("Hello!")

print(response.content)

Hi there! How can I help you today?


### State - Global State to pass the data between all the nodes

2.1 Bilding the Graph Using Langgraph

In [10]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# 1. State
class State(TypedDict):
    message: str


# 2. Node
def hello_node(state: State):
    return {
        "message": "Hello " + state["message"]
    }


# 3. Build Graph
builder = StateGraph(State)

builder.add_node("hello", hello_node)

builder.add_edge(START, "hello")
builder.add_edge("hello", END)


# 4. Compile
graph = builder.compile()


# 5. Run
result = graph.invoke({
    "message": "Mehedi"
})

print(result)

/Users/muftimehedi/Documents/Dev/langgraph-course/env/lib/python3.9/site-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


{'message': 'Hello Mehedi'}


### 2.2 Multi Node Graph

In [11]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# 1. State
class State(TypedDict):
    number: int


# 2. Node 1
def add_five(state: State):
    return {
        "number": state["number"] + 5
    }


# 3. Node 2
def multiply_two(state: State):
    return {
        "number": state["number"] * 2
    }


# 4. Build Graph
builder = StateGraph(State)

builder.add_node("add_five", add_five)
builder.add_node("multiply_two", multiply_two)

builder.add_edge(START, "add_five")
builder.add_edge("add_five", "multiply_two")
builder.add_edge("multiply_two", END)


# 5. Compile
graph = builder.compile()


# 6. Run
result = graph.invoke({
    "number": 10
})

print(result)

{'number': 30}


3. `add_messages` reducer - for chat history

In [12]:
from typing import Annotated
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages

from langchain_core.messages import HumanMessage, AIMessage


# 1. State
class State(TypedDict):
    messages: Annotated[list, add_messages]


# 2. Node
def chatbot(state: State):
    return {
        "messages": [
            AIMessage(content="Hello! How can I help you?")
        ]
    }


# 3. Build Graph
builder = StateGraph(State)

builder.add_node("chatbot", chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)


# 4. Compile
graph = builder.compile()


# 5. Run
result = graph.invoke({
    "messages": [
        HumanMessage(content="Hello")
    ]
})


# 6. Print chat history
for message in result["messages"]:
    print(message.type, ":", message.content)

human : Hello
ai : Hello! How can I help you?


### 4. Routing

In [13]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# 1. State
class State(TypedDict):
    operation: str
    number: int


# 2. Nodes
def add_node(state: State):
    return {
        "number": state["number"] + 10
    }


def multiply_node(state: State):
    return {
        "number": state["number"] * 10
    }


# 3. Router
def router(state: State):
    if state["operation"] == "add":
        return "add"
    else:
        return "multiply"


# 4. Build Graph
builder = StateGraph(State)

builder.add_node("add", add_node)
builder.add_node("multiply", multiply_node)


# 5. Routing
builder.add_conditional_edges(
    START,
    router,
    {
        "add": "add",
        "multiply": "multiply"
    }
)

builder.add_edge("add", END)
builder.add_edge("multiply", END)


# 6. Compile
graph = builder.compile()


# 7. Run
result = graph.invoke({
    "operation": "add",
    "number": 5
})

print(result)

{'operation': 'add', 'number': 15}


### 5. Memory - Checkpointer

### 6. Agents and Tools

### 7. Human in the Loop

In [14]:
from typing_extensions import TypedDict

from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver


# 1. State
class State(TypedDict):
    task: str
    approved: bool


# 2. Human approval node
def ask_approval(state: State):

    answer = interrupt(
        f"Do you approve this task: {state['task']}?"
    )

    return {
        "approved": answer
    }


# 3. Build Graph
builder = StateGraph(State)

builder.add_node("ask_approval", ask_approval)

builder.add_edge(START, "ask_approval")
builder.add_edge("ask_approval", END)


# 4. Checkpointer
memory = InMemorySaver()

graph = builder.compile(
    checkpointer=memory
)


# 5. Thread ID
config = {
    "configurable": {
        "thread_id": "conversation-1"
    }
}


# 6. Start Graph
result = graph.invoke(
    {
        "task": "Send the email",
        "approved": False
    },
    config=config
)

print(result)

{'task': 'Send the email', 'approved': False, '__interrupt__': [Interrupt(value='Do you approve this task: Send the email?', id='e660e897edab7eb1b78343f2c626b1fc')]}


### 8. Parallel Operations / Parallel Nodes

In [15]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


# 1. State
class State(TypedDict):
    name: str
    greeting: str
    question: str
    final: str


# 2. Parallel Node A
def create_greeting(state: State):
    return {
        "greeting": f"Hello {state['name']}"
    }


# 3. Parallel Node B
def create_question(state: State):
    return {
        "question": f"How are you, {state['name']}?"
    }


# 4. Final Node
def combine(state: State):
    return {
        "final": state["greeting"] + " - " + state["question"]
    }


# 5. Build Graph
builder = StateGraph(State)

builder.add_node("greeting", create_greeting)
builder.add_node("question", create_question)
builder.add_node("combine", combine)


# Parallel edges
builder.add_edge(START, "greeting")
builder.add_edge(START, "question")


# Join
builder.add_edge("greeting", "combine")
builder.add_edge("question", "combine")

builder.add_edge("combine", END)


# 6. Compile
graph = builder.compile()


# 7. Run
result = graph.invoke({
    "name": "Mehedi"
})

print(result["final"])

Hello Mehedi - How are you, Mehedi?
